In [ ]:
# Module C — Pre-Training Interventions (Towards Reasoning Equity)
# -------------------------------------------------------------------------
# This notebook illustrates how multilingual pre-training decisions can
# influence reasoning equity and language stability across languages.
#
# It provides a reproducible, didactic example of continual multilingual
# pre-training and drift monitoring, using a small real corpus from
# the Hugging Face dataset `tatoeba`.
#
# Two alternative back-ends are included:
#   Full  → a quick masked-language-model fine-tuning (Tiny BERT)
#   Fallback → a lightweight character 3-gram model that computes
#                per-language negative-log-likelihood curves.
#
# The experiment tracks per-language loss across three simulated phases
# (balanced → English-skewed → balanced) to visualise cross-language drift.
# -------------------------------------------------------------------------

In [ ]:
# 1 · Load Corpus from Tatoeba + Setup
FULL_MODE = False          # ← set True to use Transformers MLM
MODEL_NAME = "prajjwal1/bert-tiny"
LANGS = ["en", "it", "es", "zh"]
PHASES = ["phase1", "phase2", "phase3"]
SAMPLES_PER_LANG = 150

from datasets import load_dataset
import random, pandas as pd

try:
    ds = load_dataset("tatoeba", split="train")
    print("Loaded Tatoeba dataset with", len(ds), "sentences")
except Exception as e:
    print("Could not download Tatoeba → fallback toy texts:", e)
    ds = None

corpora = {}
if ds is not None:
    for L in LANGS:
        subset = ds.filter(lambda ex: ex["lang"] == L).shuffle(seed=42).select(range(min(SAMPLES_PER_LANG, 500)))
        corpora[L] = [ex["sentence"] for ex in subset]
else:
    corpora = {
        "en": ["The river bends near the old mill."]*50,
        "it": ["Il fiume curva vicino al vecchio mulino."]*50,
        "es": ["El río gira cerca del viejo molino."]*50,
        "zh": ["河流在老磨坊附近转弯。"]*50,
    }

print({k: len(v) for k,v in corpora.items()})

In [ ]:
# 2 · Sampling Policies
def sample_batches(policy:str, k:int=100):
    if policy=="round_robin":
        out=[]; [out.extend(random.sample(corpora[L], min(k,len(corpora[L])))) for L in LANGS]; return out
    if policy=="skew_to_en":
        return random.sample(corpora["en"],min(k*2,len(corpora["en"]))) + \
               random.sample(corpora["it"],min(k//2,len(corpora["it"]))) + \
               random.sample(corpora["es"],min(k//2,len(corpora["es"]))) + \
               random.sample(corpora["zh"],min(k//2,len(corpora["zh"])))
    return sum((random.sample(corpora[L],min(k,len(corpora[L]))) for L in LANGS), [])

policies = ["round_robin","skew_to_en","round_robin"]
print("Phase policies:", list(zip(PHASES, policies)))


In [ ]:
## 3 · Backend A — Tiny BERT Masked Language Model (Optional)
import math

if FULL_MODE:
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling
    from torch.utils.data import DataLoader
    import torch
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    mdl = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mdl.to(device)
    collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=0.15)
    print("Loaded tiny BERT MLM model.")


In [ ]:
def train_one_phase_MLM(texts, steps=25, batch_size=8):
    from datasets import Dataset
    from torch.utils.data import DataLoader
    ds = Dataset.from_dict({"text": texts})
    def tok_fn(ex): return tok(ex["text"], truncation=True, padding=True, max_length=64)
    ds = ds.map(tok_fn, batched=True, remove_columns=["text"])
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=collator)
    optim = torch.optim.AdamW(mdl.parameters(), lr=1e-4)
    mdl.train()
    it=0
    for batch in dl:
        for k in batch: batch[k]=batch[k].to(device)
        loss=mdl(**batch).loss
        loss.backward(); optim.step(); optim.zero_grad()
        it+=1
        if it>=steps: break

def eval_lang_loss(texts):
    from datasets import Dataset
    from torch.utils.data import DataLoader
    ds = Dataset.from_dict({"text": texts[:30]})
    def tok_fn(ex): return tok(ex["text"], truncation=True, padding=True, max_length=64)
    ds = ds.map(tok_fn, batched=True, remove_columns=["text"])
    dl = DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collator)
    mdl.eval(); tot=0; c=0
    with torch.no_grad():
        for batch in dl:
            for k in batch: batch[k]=batch[k].to(device)
            tot+=mdl(**batch).loss.item(); c+=1
    return tot/max(1,c)


In [ ]:
logs=[]
if FULL_MODE:
    for phase,policy in zip(PHASES,policies):
        print(f"--- {phase} ({policy})")
        texts=sample_batches(policy, k=60)
        train_one_phase_MLM(texts)
        for L in LANGS:
            loss=eval_lang_loss(corpora[L])
            logs.append({"phase":phase,"policy":policy,"lang":L,"loss":loss})
    df_logs=pd.DataFrame(logs)
    display(df_logs.head())


In [ ]:
## 4 · Backend B — Fallback Character n-gram Language Model
from collections import Counter
class CharNGramLM:
    def __init__(self,n=3):
        self.n=n; self.c_ng=Counter(); self.c_ctx=Counter(); self.vocab=set()
    def update(self,texts):
        for t in texts:
            s="~"*(self.n-1)+t+"$"
            for i in range(len(s)-self.n+1):
                ng=s[i:i+self.n]; ctx,ch=ng[:-1],ng[-1]
                self.c_ng[ng]+=1; self.c_ctx[ctx]+=1; self.vocab.add(ch)
    def nll(self,text):
        s="~"*(self.n-1)+text+"$"; V=max(1,len(self.vocab))
        tot,N=0.0,0
        for i in range(len(s)-self.n+1):
            ng=s[i:i+self.n]; ctx,ch=ng[:-1],ng[-1]
            num=self.c_ng[ng]+1; den=self.c_ctx[ctx]+V
            tot += - math.log(num/den); N += 1
        return tot/max(1,N)


In [ ]:
## 5 · Drift Dashboard
import matplotlib.pyplot as plt
def plot_drift(df):
    for L in LANGS:
        d=df[df["lang"]==L]
        plt.figure(figsize=(5,3))
        plt.plot(d["phase"], d["loss"], marker="o")
        plt.title(f"Per-language loss across phases — {L}")
        plt.xlabel("Phase"); plt.ylabel("Loss (MLM or NLL)")
        plt.grid(True); plt.tight_layout()
    print("Plots generated.")

plot_drift(df_logs)
df_logs.to_csv("drift_logs.csv",index=False)
print("Logs saved to drift_logs.csv")


In [ ]:
## 6 · Combined Drift Plot (Aggregated Comparison)

#To better visualise multilingual balance, we combine all languages into a single plot with distinct colours.  
#Each line represents one language, showing how its loss changes across training phases.


import seaborn as sns

def plot_combined_drift(df):
    plt.figure(figsize=(7,4))
    sns.lineplot(data=df, x="phase", y="loss", hue="lang", marker="o")
    plt.title("Cross-language Loss Drift (Combined View)")
    plt.xlabel("Training Phase")
    plt.ylabel("Loss (MLM or NLL)")
    plt.grid(True, alpha=0.3)
    plt.legend(title="Language", loc="upper right")
    plt.tight_layout()
    plt.show()

plot_combined_drift(df_logs)